In [0]:
airbnb_guests = spark.read.csv('/Volumes/workspace/stratascratch/stratascratch/guests.csv', header=True)
# display(airbnb_guests)
airbnb_hosts = spark.read.csv('/Volumes/workspace/stratascratch/stratascratch/hosts.csv', header=True)
# display(airbnb_hosts)

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window 

result = airbnb_guests.join(airbnb_hosts, 
                            (airbnb_guests['nationality'] == airbnb_hosts['nationality']) & 
                            (airbnb_guests['gender'] == airbnb_hosts['gender']), 
                            'inner')

final_result = result.select('host_id', 'guest_id').distinct().orderBy('host_id', 'guest_id')

display(final_result)

host_id,guest_id
0,9
1,5
10,6
11,11
2,1
3,7
4,0
5,2
6,4
7,10


In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

yelp_reviews = spark.read.csv('/Volumes/workspace/stratascratch/stratascratch/yelp_reviews.csv', header=True)
# yelp_reviews.show()

WindowSpec = Window.orderBy(desc(col('funny')))
yelp_reviews1 = yelp_reviews.withColumn('rnk', dense_rank().over(WindowSpec))
yelp_reviews2 = yelp_reviews1.filter(col('rnk') == 2)
yelp_reviews3 = yelp_reviews2.select('business_name', 'review_text')
display(yelp_reviews3)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


business_name,review_text
Flancer's Cafe,"This place has decent food, cute atmosphere, but the service is problematic. I was stuck in Mesa for training and had lunch there on Halloween. My pal"
Roka Akor,"I hate to admit it, but it had been a long while since my last visit to Roka Akor. I deserve a hand slap. But last week, I had the perfect excuse to p"


In [0]:
events = spark.read.csv('/Volumes/workspace/stratascratch/stratascratch/events.csv', header=True)
display(events, showtruncate=True)

user_id,occurred_at,event_type,event_name,location,device
6991,2014-06-09 18:26:54,engagement,home_page,United States,iphone 5
18851,2014-08-29 13:18:38,signup_flow,enter_info,Russia,asus chromebook
14998,2014-07-01 12:47:56,engagement,login,France,hp pavilion desktop
8186,2014-05-23 10:44:16,engagement,home_page,Italy,macbook pro
9626,2014-07-31 17:15:14,engagement,login,Russia,nexus 7
16460,2014-07-24 18:43:19,signup_flow,create_user,United States,samsung galaxy note
10101,2014-08-27 05:54:28,engagement,home_page,Singapore,dell inspiron notebook
2670,2014-05-10 10:03:34,engagement,like_message,United States,nexus 7
8708,2014-05-26 10:42:12,engagement,send_message,Australia,macbook pro
167,2014-07-30 19:39:13,engagement,view_inbox,United Arab Emirates,lenovo thinkpad


In [0]:
events = spark.read.format('csv').option('header', 'true').option('inferschema', 'true').option('mode', 'PERMISSIVE').load('/Volumes/workspace/stratascratch/stratascratch/events.csv')

#bronze_state = raw_data
# display(events, showtruncate=True) 

from pyspark.sql.functions import *
from pyspark.sql.window import Window
import re 

#silver_state = clean-up all null values and negative ids, and remove duplicates
events_silver = events.filter((col('user_id').isNotNull()) & (expr("try_cast(user_id as int) > 0"))).dropDuplicates()
# display(events_silver)

#gold_state
WindowSpec = Window.partitionBy('location')
events_gold1 = events_silver.filter(col('device').rlike('^samsung'))\
    .withColumn('number_of_users', count('user_id').over(WindowSpec))
events_gold2 = events_silver.filter((col('device').rlike('^iphone')) | (col('device').rlike('^macbook')))\
    .withColumn('number_of_users', count('user_id').over(WindowSpec))

display(events_gold2)


user_id,occurred_at,event_type,event_name,location,device,number_of_users
99999,2014-07-28 17:10:00,engagement,home_page,Argentina,macbook pro,14
16170,2014-08-25 13:32:34,engagement,home_page,Argentina,macbook pro,14
12103,2014-06-19 19:45:39,engagement,home_page,Argentina,macbook pro,14
12103,2014-06-25 11:10:04,engagement,like_message,Argentina,macbook pro,14
251,2014-08-02 10:47:41,engagement,login,Argentina,macbook air,14
16170,2014-08-19 11:07:59,engagement,login,Argentina,macbook pro,14
251,2014-08-06 15:24:42,engagement,login,Argentina,macbook air,14
16170,2014-08-23 18:53:20,engagement,like_message,Argentina,macbook pro,14
12103,2014-06-25 11:08:39,engagement,like_message,Argentina,macbook pro,14
12103,2014-06-25 11:07:03,engagement,login,Argentina,macbook pro,14


In [0]:
department_data = [
    (1, "IT"),
    (2, "Sales")
]

department_schema = ["id", "name"]

employee_data = [
    (1, "Joe", 85000, 1),
    (2, "Henry", 80000, 2),
    (3, "Sam", 60000, 2),
    (4, "Max", 90000, 1),
    (5, "Janet", 69000, 1),
    (6, "Randy", 85000, 1),
    (7, "Will", 70000, 1)
]

employee_schema = ["id", "name", "salary", "departmentId"]

from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

department = spark.createDataFrame(department_data, department_schema)
employee = spark.createDataFrame(employee_data, employee_schema)

WindowSpec1 = Window.partitionBy('departmentId').orderBy(desc('salary'))

pre_result = employee.join(department, employee['departmentId'] == department['id'], 'left').withColumn('rnk', dense_rank().over(WindowSpec1))
result = pre_result.filter(col('rnk')<=3).select(employee['name'].alias('Employee'), department['name'].alias('Department'), employee['salary'].alias('Salary')).orderBy('Department', desc('salary'), 'Employee')
display(result)

Employee,Department,Salary
Max,IT,90000
Joe,IT,85000
Randy,IT,85000
Will,IT,70000
Henry,Sales,80000
Sam,Sales,60000


In [0]:
person_data = [
    (1, "john@example.com"),
    (2, "bob@example.com"),
    (3, "john@example.com"),
    (4, "alice@example.com"),
    (5, "bob@example.com"),
    (6, "john@example.com"),
    (7, "charlie@example.com"),
    (8, "alice@example.com"),
    (9, "david@example.com"),
    (10, "bob@example.com"),
    (11, "charlie@example.com"),
    (12, "john@example.com"),
    (13, "eve@example.com"),
    (14, "alice@example.com"),
    (15, "david@example.com"),
    (16, "john@example.com"),
]

person_schema = ["id", "email"]

from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

person = spark.createDataFrame(person_data, person_schema)
# display(person)
WindowSpec = Window.partitionBy('email').orderBy('id')
person1 = person.withColumn('rnk', rank().over(WindowSpec))
person2 = person1.filter(col('rnk') == 1)
display(person2.select('id', 'email').orderBy('id'))


id,email
1,john@example.com
2,bob@example.com
4,alice@example.com
7,charlie@example.com
9,david@example.com
13,eve@example.com


In [0]:
from pyspark.sql.functions import *

us_high_growth_micro_small_cap_stocks = spark.read.option("multiLine", "true").json("/Volumes/workspace/stratascratch/usstockmarket/us_high_growth_micro_small_cap_stocks.json")

display(us_high_growth_micro_small_cap_stocks)

analyst_consensus,analyst_price_target,analyst_reviews,currency,exchange,growth_opportunity,investment_role,long_term_thesis,market_cap_category,notes,number_of_buy_signal,number_of_hold_signals,number_of_sell_signal,risk_level,sector,stock_name,theme,ticker
Buy,79.87,Buy consensus,USD,NASDAQ,Very High,High-growth space/defense core,"Neutron, launch services, satellite systems and defense contracts could materially expand Rocket Lab's addressable market.",Mid Cap,No longer a micro/small cap; it has grown substantially.,26,null,null,Very High,Aerospace & Defense,Rocket Lab Corporation,"Space launch, satellites, defense systems",RKLB
null,null,Strong institutional interest,null,NASDAQ,Very High,AI infrastructure growth,"AI data centers require rapidly increasing bandwidth and connectivity, supporting demand for high-speed connectivity solutions.",null,null,null,null,null,High,Semiconductors,Credo Technology Group Holding Ltd,AI data-center connectivity,CRDO
null,null,Needs continuous monitoring,null,NASDAQ,Very High,High-risk AI infrastructure satellite,Rising data-center bandwidth requirements can drive demand for optical components and networking products.,null,Highly volatile; position sizing matters.,null,null,null,Very High,Communications Equipment,Applied Optoelectronics Inc.,Optical networking and AI data centers,AAOI
null,null,Growth-focused coverage,null,NASDAQ,Very High,10-year asymmetric growth bet,Commercial autonomous trucking could create a large software/service market with potential fleet productivity and operating-cost benefits.,null,null,null,null,null,Very High,Autonomous Vehicles,Aurora Innovation Inc.,Autonomous trucking,AUR
Buy,84.2,Buy consensus,USD,NYSE,Very High,Speculative nuclear/AI-power moonshot,"Growing electricity demand, including from data centers, could create a major market opportunity for advanced nuclear generation.",null,Analyst counts vary by provider and date; values reflect a recent 2026 source.,15,9,1,Extreme,Nuclear Energy,Oklo Inc.,Advanced nuclear power for rising electricity demand,OKLO
null,null,Clinical-outcome dependent,null,NASDAQ,Very High,Biotech moonshot,Successful gene-editing therapies could address diseases with significant unmet medical need and create substantial commercial opportunities.,null,null,null,null,null,Extreme,Biotechnology,Intellia Therapeutics Inc.,CRISPR gene editing,NTLA
null,null,null,null,NYSE,High,Smaller space-infrastructure alternative,Increasing government and commercial spending on space infrastructure can support long-term growth.,null,null,null,null,null,Very High,Aerospace & Defense,Redwire Corporation,Space infrastructure and defense,RDW
null,null,null,null,NASDAQ,Very High,Tiny-cap robotics optionality,Autonomous last-mile delivery could become a large robotics market if unit economics and deployment scale improve.,null,Highly speculative; suitable only for a small satellite allocation.,null,null,null,Extreme,Robotics,Serve Robotics Inc.,Autonomous delivery robots,SERV
null,null,null,null,NASDAQ,High,AI application growth,"Voice and conversational AI adoption across automotive, restaurants and enterprise applications could support significant growth.",null,Valuation and competitive intensity should be monitored.,null,null,null,Very High,Artificial Intelligence,SoundHound AI Inc.,Conversational and voice AI,SOUN
